# ✈️ Time Series Forecasting — Air Passengers

**Framework:** CRISP-DM  
**Task:** Monthly forecasting  
**Methods:** Naive, Seasonal Naive, Trend + Seasonality, Holt-Winters ETS, SARIMA

This project uses the classic monthly **AirPassengers** dataset (1949–1960) and compares multiple forecasting methods before selecting the best holdout performer.


## CRISP-DM 1 — Business Understanding

### Objective
Forecast future monthly passenger demand so operations teams can plan staffing, capacity, budgeting, and service levels.

### Business questions
- Is demand trending upward?
- Is there recurring seasonality?
- Which forecasting method performs best on unseen months?
- How much error should planners expect?
- What does the next 24 months look like?

### Success criteria
Primary model selection metric: **RMSE**.  
Supporting metrics: **MAE** and **MAPE**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression

values = [
112,118,132,129,121,135,148,148,136,119,104,118,
115,126,141,135,125,149,170,170,158,133,114,140,
145,150,178,163,172,178,199,199,184,162,146,166,
171,180,193,181,183,218,230,242,209,191,172,194,
196,196,236,235,229,243,264,272,237,211,180,201,
204,188,235,227,234,264,302,293,259,229,203,229,
242,233,267,269,270,315,364,347,312,274,237,278,
284,277,317,313,318,374,413,405,355,306,271,306,
315,301,356,348,355,422,465,467,404,347,305,336,
340,318,362,348,363,435,491,505,404,359,310,337,
360,342,406,396,420,472,548,559,463,407,362,405,
417,391,419,461,472,535,622,606,508,461,390,432
]
dates = pd.date_range("1949-01-01", periods=len(values), freq="MS")
df = pd.DataFrame({"Month":dates,"Passengers":values})
display(df.head())
print(df.shape)


## CRISP-DM 2 — Data Understanding

The series contains:
- 144 monthly observations
- strong long-term growth
- recurring annual seasonality
- increasing seasonal amplitude over time

We visualize the trend and seasonal behavior before modeling.


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df["Month"],df["Passengers"])
plt.title("Monthly Airline Passengers")
plt.xlabel("Month")
plt.ylabel("Passengers")
plt.tight_layout()
plt.show()

print(df["Passengers"].describe())


In [ ]:
seasonal = df.assign(month=df["Month"].dt.month).groupby("month")["Passengers"].mean()
display(seasonal.to_frame("average_passengers"))


## CRISP-DM 3 — Data Preparation

Time-series validation must preserve chronology.

- Train: first 120 months
- Test: final 24 months
- No random shuffle
- Forecast horizon: 24 months

This prevents future information from leaking into model training.


In [ ]:
train = df.iloc[:-24].copy()
test = df.iloc[-24:].copy()

print("Train:", train["Month"].min(), "→", train["Month"].max(), len(train))
print("Test :", test["Month"].min(), "→", test["Month"].max(), len(test))


## CRISP-DM 4 — Modeling

We compare five popular approaches:

1. **Naive** — next value equals last observed value.
2. **Seasonal Naive** — next month equals the same month last year.
3. **Trend + Seasonality Regression** — linear trend plus month indicators.
4. **Holt-Winters ETS** — exponential smoothing with multiplicative trend/seasonality.
5. **SARIMA** — autoregressive integrated moving-average model with seasonal terms.


In [ ]:
import math

naive = np.repeat(train["Passengers"].iloc[-1], len(test))

seasonal_naive = np.tile(
    train["Passengers"].iloc[-12:].to_numpy(),
    math.ceil(len(test)/12)
)[:len(test)]

tmp = train.copy()
tmp["t"] = np.arange(len(tmp))
tmp["month_num"] = tmp["Month"].dt.month
X = pd.get_dummies(tmp[["t","month_num"]],columns=["month_num"],drop_first=True)

lr = LinearRegression().fit(X,tmp["Passengers"])

tt = test.copy()
tt["t"] = np.arange(len(train),len(train)+len(test))
tt["month_num"] = tt["Month"].dt.month
Xt = pd.get_dummies(tt[["t","month_num"]],columns=["month_num"],drop_first=True)
Xt = Xt.reindex(columns=X.columns,fill_value=0)
trend_seasonal = lr.predict(Xt)

preds = {
    "Naive":naive,
    "Seasonal Naive":seasonal_naive,
    "Trend + Seasonality":trend_seasonal
}


In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

ets = ExponentialSmoothing(
    train["Passengers"].astype(float),
    trend="mul",
    seasonal="mul",
    seasonal_periods=12,
    initialization_method="estimated"
).fit()
preds["Holt-Winters ETS"] = ets.forecast(len(test)).to_numpy()

sarima = SARIMAX(
    train["Passengers"].astype(float),
    order=(1,1,1),
    seasonal_order=(1,1,1,12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)
preds["SARIMA"] = sarima.forecast(len(test)).to_numpy()


## CRISP-DM 5 — Evaluation

We compare each model on the same 24-month holdout.

- **MAE:** average absolute error
- **RMSE:** penalizes large misses more strongly
- **MAPE:** average percentage error

A strong model should beat the naive and seasonal-naive baselines.


In [ ]:
def mape(y_true,y_pred):
    y_true=np.asarray(y_true,float)
    y_pred=np.asarray(y_pred,float)
    return np.mean(np.abs((y_true-y_pred)/y_true))*100

rows=[]
for name,p in preds.items():
    rows.append({
        "Model":name,
        "MAE":mean_absolute_error(test["Passengers"],p),
        "RMSE":mean_squared_error(test["Passengers"],p)**0.5,
        "MAPE":mape(test["Passengers"],p)
    })

results = pd.DataFrame(rows).sort_values("RMSE")
display(results)


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(train["Month"],train["Passengers"],label="Train")
plt.plot(test["Month"],test["Passengers"],label="Actual")

for name,p in preds.items():
    plt.plot(test["Month"],p,label=name)

plt.title("Forecast Model Comparison")
plt.xlabel("Month")
plt.ylabel("Passengers")
plt.legend(ncol=2)
plt.tight_layout()
plt.show()


## Final 24-Month Forecast

The best holdout model is refit on all available historical data, then used to forecast the next 24 months.


In [ ]:
best_name = results.iloc[0]["Model"]
print("Best model:",best_name)


## CRISP-DM 6 — Deployment & Monitoring

A forecasting model should be treated as an operating system, not a one-time chart.

### Deployment cycle
1. refresh data monthly;
2. generate the next forecast horizon;
3. compare previous forecasts with actuals;
4. monitor RMSE / MAE / MAPE;
5. compare against seasonal-naive control;
6. retrain when trend or seasonality changes.

### Dashboard KPIs
- latest actual value
- next-month forecast
- forecast horizon
- best model
- holdout MAPE
- peak future forecast


# Final Conclusion

The series has both a strong upward trend and clear yearly seasonality. A forecasting workflow should therefore use chronological validation, retain baseline forecasts, and continuously monitor forecast error after deployment.

CRISP-DM remains useful because forecasting is not only about fitting a model; it includes business objectives, data quality, time-aware validation, evaluation, deployment, and monitoring.
